# LiH Reproducibility Benchmark

This notebook extends the reproducibility benchmark pattern beyond `H2` onto the same active-space `LiH` problem already used in the scaling and cross-method notebooks.

It measures three things on one shared `LiH` problem:

- per-seed energy spread across methods
- noisy vs noiseless variance where supported
- cache-hit vs forced-rerun timing

It uses only the packaged public APIs: `run_vqe`, `run_qite`, and `run_qpe`.

The package now treats cache records without stored runtime metadata as stale and recomputes them automatically, so the main benchmark cells can safely use `force=False`.

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from common import summarize_problem, summary_stats, timed_call
from common.paths import results_dir
from qite import run_qite
from qpe import run_qpe
from vqe import run_vqe

PROBLEM_SPEC = {
    "molecule": "LiH",
    "mapping": "jordan_wigner",
    "active_electrons": 2,
    "active_orbitals": 2,
}
SEEDS = [0, 1, 2, 3]
NOISY_DEP = 0.05

problem_summary = summarize_problem(**PROBLEM_SPEC)
EXACT_GROUND = float(problem_summary["exact_ground_energy"])
NUM_QUBITS = int(problem_summary["num_qubits"])
HAMILTONIAN_TERMS = int(problem_summary["hamiltonian_terms"])


def recent_artifacts(kind, limit=3):
    root = results_dir(kind)
    files = sorted(root.glob("*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    notes = []
    for path in files[:limit]:
        payload = json.loads(path.read_text())
        notes.append(
            {
                "kind": kind,
                "file": path.name,
                "keys": ", ".join(sorted(payload.keys())[:8]),
                "size_bytes": path.stat().st_size,
            }
        )
    return notes


pd.DataFrame(
    {
        "setting": [
            "molecule",
            "mapping",
            "active_electrons",
            "active_orbitals",
            "num_qubits",
            "hamiltonian_terms",
            "exact_ground_energy",
            "num_seeds",
            "noisy_dep",
        ],
        "value": [
            PROBLEM_SPEC["molecule"],
            PROBLEM_SPEC["mapping"],
            PROBLEM_SPEC["active_electrons"],
            PROBLEM_SPEC["active_orbitals"],
            NUM_QUBITS,
            HAMILTONIAN_TERMS,
            EXACT_GROUND,
            len(SEEDS),
            NOISY_DEP,
        ],
    }
)


## Seed spread with cache-aware reruns

This section uses `force=False` and records `compute_runtime_s` when available.

If a cached artifact predates runtime metadata, the package will refresh it automatically.

In [ ]:
records = []

for seed in SEEDS:
    vqe_base = dict(
        **PROBLEM_SPEC,
        ansatz_name="UCCSD",
        optimizer_name="Adam",
        steps=75,
        stepsize=None,
        plot=False,
        seed=seed,
    )
    for noisy, dep in [(False, 0.0), (True, NOISY_DEP)]:
        res, runtime_s = timed_call(
            run_vqe,
            **vqe_base,
            noisy=noisy,
            depolarizing_prob=dep,
            force=False,
        )
        records.append(
            {
                "method": "VQE",
                "variant": "noisy" if noisy else "noiseless",
                "seed": int(seed),
                "energy": float(res["energy"]),
                "abs_error": abs(float(res["energy"]) - EXACT_GROUND),
                "runtime_s": float(res.get("compute_runtime_s", runtime_s)),
                "compute_runtime_s": float(res.get("compute_runtime_s", runtime_s)),
                "cache_hit": bool(res.get("cache_hit", False)),
                "num_qubits": int(res["num_qubits"]),
            }
        )

    qite_res, runtime_s = timed_call(
        run_qite,
        **PROBLEM_SPEC,
        ansatz_name="UCCSD",
        steps=75,
        dtau=0.2,
        plot=False,
        show=False,
        seed=seed,
        force=False,
    )
    records.append(
        {
            "method": "VarQITE",
            "variant": "noiseless",
            "seed": int(seed),
            "energy": float(qite_res["energy"]),
            "abs_error": abs(float(qite_res["energy"]) - EXACT_GROUND),
            "runtime_s": float(qite_res.get("compute_runtime_s", runtime_s)),
            "compute_runtime_s": float(qite_res.get("compute_runtime_s", runtime_s)),
            "cache_hit": bool(qite_res.get("cache_hit", False)),
            "num_qubits": int(qite_res["num_qubits"]),
        }
    )

    qpe_base = dict(
        **PROBLEM_SPEC,
        n_ancilla=4,
        t=1.0,
        trotter_steps=2,
        shots=1000,
        plot=False,
        seed=seed,
    )
    for noisy, dep in [(False, 0.0), (True, NOISY_DEP)]:
        res, runtime_s = timed_call(
            run_qpe,
            **qpe_base,
            noisy=noisy,
            depolarizing_prob=dep,
            force=False,
        )
        records.append(
            {
                "method": "QPE",
                "variant": "noisy" if noisy else "noiseless",
                "seed": int(seed),
                "energy": float(res["energy"]),
                "abs_error": abs(float(res["energy"]) - EXACT_GROUND),
                "runtime_s": float(res.get("compute_runtime_s", runtime_s)),
                "compute_runtime_s": float(res.get("compute_runtime_s", runtime_s)),
                "cache_hit": bool(res.get("cache_hit", False)),
                "num_qubits": int(res["num_qubits"]),
            }
        )

records_df = pd.DataFrame(records)
display(records_df.round(8))


In [ ]:
grouped = defaultdict(list)
for row in records:
    grouped[(row["method"], row["variant"])].append(row)

summary = []
for (method, variant), rows in sorted(grouped.items()):
    energies = [r["energy"] for r in rows]
    errors = [r["abs_error"] for r in rows]
    runtimes = [r["runtime_s"] for r in rows]
    summary.append(
        {
            "method": method,
            "variant": variant,
            "energy_mean": summary_stats(energies)["mean"],
            "energy_std": summary_stats(energies)["std"],
            "abs_error_mean": summary_stats(errors)["mean"],
            "runtime_mean_s": summary_stats(runtimes)["mean"],
        }
    )

summary_df = pd.DataFrame(summary).sort_values(["abs_error_mean", "runtime_mean_s"])
display(summary_df.round(8))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for variant in ["noiseless", "noisy"]:
    subset = records_df[records_df["variant"] == variant]
    if subset.empty:
        continue
    for method in subset["method"].unique():
        rows = subset[subset["method"] == method].sort_values("seed")
        label = f"{method} ({variant})"
        axes[0].plot(rows["seed"], rows["energy"], marker="o", label=label)
        axes[1].plot(rows["seed"], rows["abs_error"], marker="o", label=label)

axes[0].axhline(EXACT_GROUND, color="black", linestyle="--", linewidth=1, label="Exact ground")
axes[0].set_title("Energy by Seed")
axes[0].set_xlabel("Seed")
axes[0].set_ylabel("Energy (Ha)")
axes[0].grid(alpha=0.3)

axes[1].set_title("Absolute Error by Seed")
axes[1].set_xlabel("Seed")
axes[1].set_ylabel("|E - E_exact| (Ha)")
axes[1].grid(alpha=0.3)

handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=False)
plt.tight_layout()
plt.show()


## Cache-hit versus forced-rerun timing

This section measures the gap between:

- a forced compute (`force=True`)
- a cache-eligible reload (`force=False`)

using one representative seed per method.

In [ ]:
timing_rows = []
seed = 0

cases = [
    (
        "VQE",
        run_vqe,
        dict(
            **PROBLEM_SPEC,
            ansatz_name="UCCSD",
            optimizer_name="Adam",
            steps=75,
            stepsize=None,
            plot=False,
            seed=seed,
            noisy=False,
        ),
    ),
    (
        "VarQITE",
        run_qite,
        dict(
            **PROBLEM_SPEC,
            ansatz_name="UCCSD",
            steps=75,
            dtau=0.2,
            plot=False,
            show=False,
            seed=seed,
        ),
    ),
    (
        "QPE",
        run_qpe,
        dict(
            **PROBLEM_SPEC,
            n_ancilla=4,
            t=1.0,
            trotter_steps=2,
            shots=1000,
            plot=False,
            seed=seed,
            noisy=False,
        ),
    ),
]

for method, fn, kwargs in cases:
    forced_res, forced_elapsed = timed_call(fn, **kwargs, force=True)
    cached_res, cached_elapsed = timed_call(fn, **kwargs, force=False)
    timing_rows.append(
        {
            "method": method,
            "forced_elapsed_s": float(forced_elapsed),
            "forced_compute_runtime_s": float(forced_res.get("compute_runtime_s", forced_elapsed)),
            "cached_elapsed_s": float(cached_elapsed),
            "cached_compute_runtime_s": float(cached_res.get("compute_runtime_s", cached_elapsed)),
            "cached_cache_hit": bool(cached_res.get("cache_hit", False)),
        }
    )

timing_df = pd.DataFrame(timing_rows)
display(timing_df.round(8))


In [ ]:
artifact_df = pd.DataFrame(
    recent_artifacts("vqe") + recent_artifacts("qite") + recent_artifacts("qpe")
)
artifact_df


## Interpretation

- If `LiH` keeps small seed spread under the current active-space setup, the package results remain decision-grade beyond `H2`.
- If `QPE` shows much larger spread or error than the variational methods, that strengthens the case for the remaining QPE calibration follow-on notebook.
- If `cache_hit=True` collapses runtime substantially while preserving the stored `compute_runtime_s`, the current caching model remains useful for larger-system benchmark workflows.